# AP01 – Fehleranalyse

### SQL-Fehleranalyse

| Fehler / Signal | Korrektur / Begründung |
|---|---|
| Objekte ohne Treffer fehlen | `LEFT JOIN` statt `INNER JOIN` verwenden. Die vollständig benötigten Objekte stehen links, bspw. `cinemas LEFT JOIN screenings LEFT JOIN tickets` |
| Objekt ohne Treffer wird als 1 gezählt | `COUNT(tickets.ticketid)` statt `COUNT(*)`. Beim `LEFT JOIN` zählt `COUNT(*)` auch die erhaltene Zeile ohne Ticket, `COUNT(tickets.ticketid)` ergibt dafür 0 |
| „Noch nie / ohne“ liefert nicht die gesuchten Objekte | `LEFT JOIN` mit `WHERE tickets.ticketid IS NULL` verwenden. Ein `INNER JOIN` enthält nur Objekte mit passendem Treffer |
| Objekte ohne Verkäufe im gesuchten Zeitraum fehlen | Zeitfilter der rechten Tabelle in `ON` aufnehmen. Ein Filter in `WHERE` kann die erhaltenen Zeilen ohne Treffer wieder ausschliessen |
| Anzahl oder Summe ist nach mehreren JOINs zu hoch | Prüfen, ob mehrere passende Datensätze Zeilen vervielfachen. Eindeutige Objekte gegebenenfalls mit `COUNT(DISTINCT ...)` zählen, Summen gegebenenfalls vor dem JOIN aggregieren |
| Falsche Werte werden gezählt | `COUNT(*)` zählt Zeilen, `COUNT(spalte)` nur vorhandene Werte und `COUNT(DISTINCT spalte)` unterschiedliche vorhandene Werte. Die Variante passend zur Aufgabe wählen |
| „Pro Person“ fasst verschiedene Personen zusammen | Zusätzlich nach der eindeutigen ID gruppieren. Gleiche Vor- und Nachnamen können zu unterschiedlichen Personen gehören |
| Durchschnitt wird über die falschen Gruppen berechnet | Gruppierung prüfen. Bspw. `GROUP BY customers.membership_type` für den Durchschnitt je Mitgliedschaftstyp |
| Durchschnitt berücksichtigt nur einen Teil der gewünschten Daten | Filter prüfen. `WHERE` schliesst Beobachtungen bereits vor der Durchschnittsberechnung aus |
| Gruppen werden anhand der falschen Bedingung ausgewählt | `WHERE` filtert einzelne Zeilen, `HAVING` die berechneten Gruppen. Bspw. einzelne Preise über 20 oder Gruppen mit Durchschnittspreis über 20 unterscheiden |
| Ungültige Kategorien mit fehlenden Werten werden übersehen | `NOT IN (...)` erfasst keine NULL-Werte. Falls fehlende Angaben ebenfalls gesucht sind, zusätzlich `OR membership_type IS NULL` verwenden |
| Ergebnis enthält zu viele oder falsche Zeilen | Kombination aus `AND` und `OR` prüfen und passend klammern. Bspw. `(Marke A OR Marke B) AND Preisbedingung` |
| Falsche Top- oder Bottom-Werte erscheinen | Sortierrichtung prüfen. Teuerste zuerst mit `DESC`, günstigste zuerst mit `ASC`. Fehlende Werte bei Bedarf mit `NULLS LAST` ans Ende setzen |
| Falscher Zeitraum wird ausgewertet | Datumsgrenzen prüfen. Für das Jahr 2023 bspw. `>= '2023-01-01' AND < '2024-01-01'` verwenden |
| Fehlende Summe erscheint als NULL statt 0 | `COALESCE(SUM(sales.saleprice), 0)` verwenden, wenn fehlende Verkäufe gemäss Aufgabe als Umsatz 0 dargestellt werden sollen |
| Durchschnitt ist verfälscht, weil fehlende Werte als 0 eingesetzt wurden | NULL nicht pauschal durch 0 ersetzen. `AVG(spalte)` ignoriert fehlende Werte, `AVG(COALESCE(spalte, 0))` bezieht sie als Nullwerte ein |

## Die zehn Kernfragen
1. Fehlen Objekte ohne Treffer? → `INNER JOIN` vs. `LEFT JOIN` prüfen, bei LEFT JOIN muss die vollständig benötigte Objekttabelle links stehen
2. Sollte eine Gruppe 0 haben, zeigt aber 1? → Bei LEFT JOIN zählt `COUNT(*)` auch die ergänzte Zeile, mit `COUNT(rechte_id)` nur echte Treffer zählen und dafür eine ID verwenden, die bei echten Treffern niemals NULL ist
3. Gesucht sind «noch nie» oder «ohne»? → `LEFT JOIN ... WHERE rechte_id IS NULL` oder `NOT EXISTS` verwenden, für die NULL-Prüfung eine rechte ID wählen, die bei echten Treffern niemals NULL ist
4. Filter auf Einzelzeilen? → `WHERE` verwenden, Filter auf Gruppenkennzahlen mit `HAVING COUNT(...)`, `HAVING AVG(...)` usw. umsetzen
5. Aggregation «pro X»? → `GROUP BY X` verwenden, Personen nach ID statt nur nach Namen gruppieren und nicht aggregierte Ausgabespalten grundsätzlich mitgruppieren
6. Spalten aus mehreren Tabellen benötigt? → JOIN-Pfad über passende Primär- und Fremdschlüssel prüfen, jede Verbindung braucht eine passende ON-Bedingung, ausser beim beabsichtigten CROSS JOIN
7. Ungültige Kategorien gesucht? → `NOT IN ('gültig1', 'gültig2')` verwenden, Gross- und Kleinschreibung sowie Leerzeichen beachten und fehlende Kategorien bei Bedarf mit `OR spalte IS NULL` erfassen
8. NULL geprüft? → `IS NULL` oder `IS NOT NULL` verwenden, niemals `= NULL`, leerer Text `''` ist kein NULL und auch `<>`, `NOT IN` und `NOT LIKE` wählen NULL nicht automatisch aus
9. Sortierung gefordert? → `ORDER BY ... ASC/DESC` verwenden, für LIMIT und OFFSET eine eindeutige Reihenfolge festlegen, bspw. `ORDER BY price DESC, bicycleid`, fehlende Werte bei Bedarf mit NULLS LAST ans Ende stellen
10. Ergebnis fachlich plausibel? → Zeilenanzahl, bekannte Objektanzahlen, Summen und Fälle mit 0 prüfen, ein leeres Ergebnis kann korrekt sein

### Wichtige Ergänzungen
11. Trotz LEFT JOIN fehlen Objekte ohne passende Verkäufe im Zeitraum? → Datumsfilter der rechten Tabelle in ON setzen, wenn alle linken Objekte erhalten bleiben sollen, derselbe Filter in WHERE entfernt die ergänzten NULL-Zeilen
12. Summe fehlt statt 0? → SUM, AVG, MIN und MAX ignorieren NULL und liefern ohne bekannte Werte NULL, falls fachlich passend `COALESCE(SUM(betrag), 0)` verwenden
13. Anzahl oder Umsatz zu hoch? → Mehrere 1:n-JOINs können Zeilen vervielfachen, Beziehungen prüfen und nötigenfalls vor dem JOIN aggregieren, `COUNT(DISTINCT id)` zählt eindeutige Objekte, `SUM(DISTINCT betrag)` ist keine allgemeine Reparatur, weil echte Geschäftsfälle denselben Betrag haben können
14. NOT IN mit Unterabfrage liefert unerwartet keine Treffer? → NULL in der Unterabfrage ausschliessen oder NOT EXISTS verwenden, bei NULL in der linken Vergleichsspalte ebenfalls die gewünschte Behandlung festlegen
15. AND und OR kombiniert? → AND bindet stärker als OR, gewünschte Logik einklammern, bspw. `WHERE (make = 'Bianchi' OR make = 'Trek') AND price > 3000`
16. Datum oder Preis an einer Grenze falsch gefiltert? → BETWEEN schliesst beide Grenzen ein, für ganze Jahre inkl. Uhrzeiten besser `datum >= '2023-01-01' AND datum < '2024-01-01'` verwenden
17. Textmuster verwechselt? → Bei LIKE steht `%` für beliebig viele Zeichen inkl. keinem Zeichen und `_` für genau ein Zeichen, ILIKE ignoriert Gross- und Kleinschreibung, bei Regex berücksichtigt `~` die Schreibweise und `~*` ignoriert sie, `^` steht für Anfang und `$` für Ende
18. Regex-Backslashes oder Prozentzeichen verursachen Probleme? → Python-Strings mit Regex-Backslashes als `r"""..."""` schreiben, im bisherigen Abfragemuster mit psycopg2 Prozentzeichen verdoppeln, bspw. `LIKE 'B%%'`

## SQL-Reihenfolge

### Schreibreihenfolge

Nur die benötigten Bestandteile verwenden

```sql
SELECT DISTINCT
    gruppenspalte,
    COUNT(*) AS anzahl,
    SUM(wertspalte) AS summe,
    ROUND(CAST(AVG(wertspalte) AS NUMERIC), 2) AS durchschnitt,
    MIN(wertspalte) AS minimum,
    MAX(wertspalte) AS maximum
FROM tabelle1 AS a
LEFT JOIN tabelle2 AS b
    ON a.id = b.fremdschluessel
WHERE bedingung
GROUP BY gruppenspalte
HAVING COUNT(*) > grenzwert
ORDER BY anzahl DESC, gruppenspalte ASC
LIMIT anzahl_zeilen
OFFSET uebersprungene_zeilen
```

| Bestandteil | Zweck |
|---|---|
| `SELECT` | Ausgabespalten und berechnete Werte bestimmen |
| `DISTINCT` | Doppelte vollständige Ergebniszeilen entfernen |
| `FROM` | Ausgangstabelle bestimmen |
| `JOIN ... ON` | Weitere Tabellen über passende Spalten verbinden |
| `WHERE` | Einzelne Zeilen vor der Aggregation filtern |
| `GROUP BY` | Eine Gruppe pro Wert oder Wertekombination bilden |
| `HAVING` | Gruppen anhand einer Bedingung filtern, bspw. anhand von COUNT oder AVG |
| `ORDER BY` | Ergebnis sortieren, je Spalte mit ASC oder DESC und bei Bedarf NULLS FIRST oder NULLS LAST |
| `LIMIT` | Ausgabe auf maximal die angegebene Zeilenanzahl begrenzen |
| `OFFSET` | Angegebene Anzahl Zeilen überspringen |

### Ausdrücke innerhalb von SELECT
- Einzelne Spalten, `*`, Berechnungen und Ausdrücke mit `AS alias`
- Aggregatfunktionen wie COUNT, SUM, AVG, MIN und MAX
- Funktionen wie ROUND, CAST, EXTRACT, COALESCE und Textfunktionen
- Fallunterscheidungen mit `CASE WHEN ... THEN ... ELSE ... END`
- Fensterfunktionen wie `AVG(...) OVER (...)`, ROW_NUMBER und RANK
- Diese Funktionen haben keine eigene feste Position zwischen den SQL-Klauseln, sie stehen innerhalb der jeweiligen Ausdrücke

### JOINs und Bedingungen
- Statt LEFT JOIN je nach Aufgabe INNER JOIN, RIGHT JOIN, FULL OUTER JOIN oder CROSS JOIN verwenden
- Mehrere JOINs hintereinander sind möglich, CROSS JOIN benötigt keine ON-Bedingung
- WHERE und HAVING können mehrere Bedingungen mit AND, OR und Klammern enthalten
- IN, BETWEEN, LIKE, Regex, EXISTS und Unterabfragen sind Bestandteile von Ausdrücken oder Bedingungen, keine zusätzlichen Schritte der Reihenfolge